# Robust Multi-Asset Allocation under Estimation Error

**Warning:** the bundled experiment uses synthetic returns solely to test the pipeline. Results are not market evidence. Read `docs/RESEARCH_NOTE.md` before interpreting outputs.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'src'))
import matplotlib.pyplot as plt
import pandas as pd
from robust_allocator.data import make_synthetic_returns
from robust_allocator.backtest import walk_forward_backtest
from robust_allocator.metrics import summarize

## 1. Generate deterministic synthetic data
The latent regime is known only because this is a simulation. In real data, regimes are unobserved and must not be treated as known labels.

In [ ]:
returns, latent_regime = make_synthetic_returns(n_periods=360, seed=7)
display(returns.head())
display(latent_regime.value_counts().rename('periods'))

## 2. Walk-forward allocation comparison
At each rebalance, estimation uses only the trailing 60 observations, ending before the return being evaluated. Transaction costs are stylized proportional costs.

In [ ]:
strategy_returns, weights = walk_forward_backtest(returns, lookback=60, rebalance_every=1, cost_bps=5)
rows = [{'strategy': name, **summarize(strategy_returns[name], weights[name])} for name in strategy_returns]
summary = pd.DataFrame(rows).set_index('strategy')
display(summary.round(4))

## 3. Plot performance and allocation stability
Do not select a strategy by the best line on this single simulated sample. Repeat over many seeds and stress settings before making comparative claims.

In [ ]:
(1 + strategy_returns).cumprod().plot(figsize=(12, 6), title='Synthetic demonstration only — not market performance')
plt.ylabel('Growth of $1'); plt.tight_layout()

In [ ]:
weights['hrp'].plot.area(figsize=(12, 6), title='HRP weights — synthetic demonstration')
plt.ylabel('Portfolio weight'); plt.tight_layout()

## 4. Research interpretation checklist
- Did shrinkage improve realized risk or just change the weight profile?
- Is HRP more stable across seeds and lookback windows?
- Do differences survive higher costs and correlation/volatility shocks?
- What is the distribution of results across seeds, rather than the best seed?
- Are findings robust to removing individual assets and changing the universe?

**No conclusion is drawn from this single synthetic run.** The next step is a pre-registered Monte Carlo grid with known covariance matrices, followed by a separate real-data study with point-in-time total-return indices.